# Load the Trained E0 ResNet50 Model — IP102

This notebook downloads and loads the **existing E0 baseline checkpoint**.

- Dataset: IP102
- Architecture: ResNet50
- Classes: 102
- Checkpoint: GitHub Release `e0-baseline-v1`
- Best epoch: 9
- Validation Macro-F1: 0.634278

**No training or fine-tuning is performed.**


## 1. Clone the GitHub repository

In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path("/content/ip102-data-cleaning-research")

if not (repo_dir / ".git").exists():
    subprocess.run(
        ["git", "clone", "https://github.com/ecalban/ip102-data-cleaning-research.git", str(repo_dir)],
        check=True
    )
else:
    print("Repository already exists:", repo_dir)

assert (repo_dir / "results/E0/test_results.csv").is_file()
print("Repository ready:", repo_dir)


## 2. Download the existing E0 checkpoint

In [ ]:
import requests

checkpoint_url = "https://github.com/ecalban/ip102-data-cleaning-research/releases/download/e0-baseline-v1/best_model.pth"
checkpoint_path = Path("/content/e0_best_model.pth")

if not checkpoint_path.exists():
    with requests.get(
        checkpoint_url,
        stream=True,
        timeout=120,
        allow_redirects=True
    ) as response:
        response.raise_for_status()

        with checkpoint_path.open("wb") as f:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    f.write(chunk)

print("Checkpoint:", checkpoint_path)
print("Size:", checkpoint_path.stat().st_size, "bytes")

expected_size = 285061172

assert checkpoint_path.stat().st_size == expected_size, (
    "Checkpoint size mismatch!"
)


## 3. Select CPU or CUDA GPU

In [ ]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA unavailable; using CPU.")


## 4. Load the checkpoint and inspect metadata

In [ ]:
checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

required_keys = {
    "epoch",
    "model_state_dict",
    "val_macro_f1"
}

assert required_keys.issubset(checkpoint.keys()), (
    "Unexpected checkpoint structure."
)

print("Checkpoint epoch:", checkpoint["epoch"])
print(
    "Validation Macro-F1:",
    checkpoint["val_macro_f1"]
)
print("Seed:", checkpoint.get("seed", "Not recorded"))


## 5. Reconstruct ResNet50 and load trained weights

In [ ]:
from torchvision import models
from torch import nn

# Same ResNet50 architecture as the original E0 experiment.
# weights=None avoids downloading ImageNet weights.
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 102)

model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

model = model.to(device)
model.eval()

print("E0 model loaded successfully.")
print("Number of output classes:", model.fc.out_features)
print("Training mode:", model.training)


## 6. Display the original E0 test results

In [ ]:
import pandas as pd

results_path = repo_dir / "results/E0/test_results.csv"

test_results = pd.read_csv(results_path)

print("Original E0 test results:")
display(test_results)


## 7. Confirm model is ready for inference

In [ ]:
assert model.training is False
assert model.fc.out_features == 102

print("E0 ResNet50 is ready for inference.")
print("Device:", device)
print("No training was performed.")
